# Clonar Repo

In [ ]:
import getpass

USER = "kassmontezuma"
REPO = "Capstone"
TOKEN = getpass.getpass('Introduce tu GitHub Personal Access Token: ')

!git clone https://{TOKEN}@github.com/{USER}/{REPO}.git

%cd {REPO}
!git checkout DeepLearning

# Librerías

In [ ]:
import os
import gc
import cv2
import math
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torchvision.models import MobileNet_V2_Weights
from torchvision.models import DenseNet121_Weights
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet18_Weights
import torchvision.models as models
from torchvision import transforms
import torch.nn as nn
import torch


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

# Configuración

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

BATCH_SIZE = 32
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4

# Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Rutas

In [ ]:
BASE_DIR = "/content/drive/MyDrive/CAPSTONE"

TRAIN_PATCH_DIR = os.path.join(BASE_DIR, "dataset_train")
VAL_PATCH_DIR = os.path.join(BASE_DIR, "dataset_validation")
TEST_PATCH_DIR = os.path.join(BASE_DIR, "dataset_test")

SAVE_DIR = os.path.join(BASE_DIR,"Modelos_DL")
!mkdir -p "{SAVE_DIR}/models"
!mkdir -p "{SAVE_DIR}/results"
!mkdir -p "{SAVE_DIR}/plots"

# Cargar dataset

In [ ]:
train_df = pd.read_csv("dataset_train.csv")
val_df = pd.read_csv("dataset_validation.csv")
test_df = pd.read_csv("dataset_test.csv")

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Corregir rutas

In [ ]:
train_df["path"] = train_df["patch"].apply(
    lambda x: os.path.join(TRAIN_PATCH_DIR, x)
)

val_df["path"] = val_df["patch"].apply(
    lambda x: os.path.join(VAL_PATCH_DIR, x)
)

test_df["path"] = test_df["patch"].apply(
    lambda x: os.path.join(TEST_PATCH_DIR, x)
)
print(train_df.iloc[0]["path"])
print(val_df.iloc[0]["path"])
print(test_df.iloc[0]["path"])

In [ ]:
class LungCancerDataset(Dataset):
    def __init__(self, dataframe, target_size=(224, 224)):
        self.df = dataframe
        self.target_size = target_size

        # Normalización para redes pre-entrenadas (ImageNet stats)
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Cargar patch 3D
        path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['label']

        #Verifica que el archivo existe
        if not os.path.exists(path):
            print(f"Archivo no encontrado: {path}")
            # Retornar un patch dummy (esto no debería pasar si las rutas están bien)
            dummy_patch = np.zeros((64, 64, 64), dtype=np.float32)
            patch = dummy_patch
        else:
            patch = np.load(path)

        # 2. Extraer slices centrales de los 3 planos
        z_center, y_center, x_center = patch.shape[0]//2, patch.shape[1]//2, patch.shape[2]//2

        axial_slice = patch[z_center, :, :]      # plano axial (horizontal)
        coronal_slice = patch[:, y_center, :]    # plano coronal (frontal)
        sagital_slice = patch[:, :, x_center]    # plano sagital (lateral)

        # 3. Convertir a uint8 (0-255) para redimensionamiento con OpenCV
        # Porque tus valores están en [0,1] después del preprocessing
        axial = (axial_slice * 255).astype(np.uint8)
        coronal = (coronal_slice * 255).astype(np.uint8)
        sagital = (sagital_slice * 255).astype(np.uint8)

        # 4. Redimensionar a target_size (224x224)
        axial_resized = cv2.resize(axial, self.target_size, interpolation=cv2.INTER_LINEAR)
        coronal_resized = cv2.resize(coronal, self.target_size, interpolation=cv2.INTER_LINEAR)
        sagital_resized = cv2.resize(sagital, self.target_size, interpolation=cv2.INTER_LINEAR)

        # 5. Normalizar a [0,1] nuevamente (para convertir a tensor después)
        axial_norm = axial_resized / 255.0
        coronal_norm = coronal_resized / 255.0
        sagital_norm = sagital_resized / 255.0

        # 6. Stack como canales RGB (3, H, W)
        image = np.stack([axial_norm, coronal_norm, sagital_norm], axis=0)  # (3, 224, 224)

        # 7. Convertir a tensor y aplicar normalización ImageNet
        image = torch.from_numpy(image).float()
        image = self.normalize(image)

        # 8. Label (para clasificación binaria)
        label = torch.tensor(float(label), dtype=torch.float32)

        return image, label

# Crear dataset

In [ ]:
train_dataset = LungCancerDataset(train_df)
val_dataset = LungCancerDataset(val_df)
test_dataset = LungCancerDataset(test_df)

# Verificar sizes
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
import zipfile
import os

BASE_DIR = "/content/drive/MyDrive/CAPSTONE"

for folder in ["dataset_validation"]:

    zip_path = os.path.join(BASE_DIR, f"{folder}.zip")
    extract_path = os.path.join(BASE_DIR, folder)

    if not os.path.exists(extract_path):
        print(f"Extrayendo {folder}...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_path)

print("Todo listo.")

# Verificar Rutas

In [ ]:
def check_paths(df, name):
    missing = 0
    for idx, row in df.iterrows():
        if not os.path.exists(row['path']):
            missing += 1
            print(f"Missing: {row['path']}")
    print(f"{name}: {missing} archivos faltantes de {len(df)}")
    return missing

print("\nVerificando rutas...")
missing_train = check_paths(train_df, "Train")
missing_val = check_paths(val_df, "Val")
missing_test = check_paths(test_df, "Test")

if missing_train + missing_val + missing_test > 0:
    print("\nHay archivos faltantes. Revisa tus rutas en Drive.")
else:
    print("\nTodas las rutas son correctas")

# Visualizar un ejemplo

In [ ]:
idx = random.randint(0, len(train_dataset)-1)
sample_image, sample_label = train_dataset[idx]
print(f"\nEjemplo de datos:")
print(f"  - Imagen shape: {sample_image.shape}")
print(f"  - Label: {sample_label}")
print(f"  - Rango: [{sample_image.min():.2f}, {sample_image.max():.2f}]")

def imshow(tensor, title=None):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    tensor = tensor * std + mean
    tensor = torch.clamp(tensor, 0, 1)
    npimg = tensor.numpy().transpose(1,2,0)
    plt.imshow(npimg)
    if title:
        plt.title(title)
    plt.axis('off')

plt.figure(figsize=(6,6))
imshow(sample_image, f"Label: {sample_label.item()}")
plt.show()

# Crear DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataLoaders creados exitosamente")
print(f"  - Batches por epoch (train): {len(train_loader)}")
print(f"  - Batches por epoch (val): {len(val_loader)}")


# Probar un batch

In [ ]:
images, labels = next(iter(train_loader))
print(f"\nBatch test:")
print(f"  - Images shape: {images.shape}")  # [32, 3, 224, 224]
print(f"  - Labels shape: {labels.shape}")  # [32]
print(f"  - Labels: {labels[:10]}")
print(f"  - Image range: [{images.min():.2f}, {images.max():.2f}]")

# Visualizar primeras 4 imágenes del batch
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_display = images[i] * std + mean
    img_display = torch.clamp(img_display, 0, 1)

    axes[i].imshow(img_display.permute(1,2,0))
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


# Sanity Check

In [ ]:
# 1. Verificar que los DataLoaders funcionan
print("\nVerificando DataLoaders...")
try:
    sample_batch_images, sample_batch_labels = next(iter(train_loader))
    print(f"   Batch shape: {sample_batch_images.shape}")
    print(f"   Labels shape: {sample_batch_labels.shape}")
    print(f"   Image range: [{sample_batch_images.min():.2f}, {sample_batch_images.max():.2f}]")
    print(f"   Label values: {torch.unique(sample_batch_labels)}")
except Exception as e:
    print(f"   Error: {e}")
    raise

# 2. Verificar distribución de clases
print("\nVerificando balance de clases...")
train_labels = [train_df.iloc[i]['label'] for i in range(len(train_df))]
val_labels = [val_df.iloc[i]['label'] for i in range(len(val_df))]
test_labels = [test_df.iloc[i]['label'] for i in range(len(test_df))]

print(f"   Train - Positivos: {sum(train_labels)}/{len(train_labels)} ({100*sum(train_labels)/len(train_labels):.1f}%)")
print(f"   Train - Negativos: {len(train_labels)-sum(train_labels)}/{len(train_labels)} ({100*(len(train_labels)-sum(train_labels))/len(train_labels):.1f}%)")
print(f"   Val   - Positivos: {sum(val_labels)}/{len(val_labels)} ({100*sum(val_labels)/len(val_labels):.1f}%)")
print(f"   Test  - Positivos: {sum(test_labels)}/{len(test_labels)} ({100*sum(test_labels)/len(test_labels):.1f}%)")

# 3. Visualización de sanity check
print("\nVisualizando ejemplo de datos...")
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Imagen de entrenamiento
    img, label = train_dataset[i]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_display = img * std + mean
    img_display = torch.clamp(img_display, 0, 1)

    axes[0, i].imshow(img_display.permute(1,2,0))
    axes[0, i].set_title(f"Train - Label: {label.item()}")
    axes[0, i].axis('off')

    # Imagen de validación
    img_val, label_val = val_dataset[i]
    img_val_display = img_val * std + mean
    img_val_display = torch.clamp(img_val_display, 0, 1)

    axes[1, i].imshow(img_val_display.permute(1,2,0))
    axes[1, i].set_title(f"Val - Label: {label_val.item()}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("\nRESUMEN:")
print(f"   - Device: {DEVICE}")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Train samples: {len(train_dataset)}")
print(f"   - Val samples: {len(val_dataset)}")
print(f"   - Test samples: {len(test_dataset)}")

print(
    f"   - Clases balanceadas: "
    f"{100*sum(train_labels)/len(train_labels):.1f}% positivos"
)

print("\nTODO LISTO PARA ENTRENAR LOS MODELOS")

# Balance de clases

In [ ]:
train_labels = train_df["label"].tolist()

num_negatives = train_labels.count(0)
num_positives = train_labels.count(1)

print(f"Negativos (0): {num_negatives}")
print(f"Positivos (1): {num_positives}")

pos_weight_value = num_negatives / num_positives

print(f"Ratio Neg/Pos : {pos_weight_value:.4f}")

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_value], device=DEVICE)
)

# Función para gráficos

In [ ]:
def generar_graficos(MODEL_NAME, train_losses, val_losses, val_accuracies,
                          test_labels, test_preds, test_probs,
                          test_acc, test_recall, test_specificity,
                          test_precision, test_f1, test_auc, cm_test,
                          SAVE_DIR):

    # Configuración para IEEE
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.size'] = 10
    plt.rcParams['axes.labelsize'] = 11
    plt.rcParams['axes.titlesize'] = 12
    plt.rcParams['legend.fontsize'] = 9
    plt.rcParams['figure.dpi'] = 300

    # Crear directorio
    model_dir = os.path.join(SAVE_DIR, 'plots', MODEL_NAME)
    os.makedirs(model_dir, exist_ok=True)

    blues = {
        'dark': '#0d3b66',      # Azul oscuro
        'medium': '#1f77b4',    # Azul medio
        'light': '#4a90e2',     # Azul claro
        'lighter': '#7bb3e8',   # Azul más claro
        'lightest': '#a8c9f0',  # Azul muy claro
        'gray': '#7f7f7f',      # Gris para referencias
        'black': '#000000'      # Negro
    }

    # GRÁFICO 1: Curvas de Pérdida
    print("Gráfico 1: Curvas de pérdida...")
    fig, ax = plt.subplots(figsize=(8, 5))
    epochs = range(1, len(train_losses) + 1)

    ax.plot(epochs, train_losses, 'o-', color=blues['medium'], linewidth=1.5,
            markersize=4, label='Training Loss', markeredgecolor='black', markeredgewidth=0.5)
    ax.plot(epochs, val_losses, 's-', color=blues['dark'], linewidth=1.5,
            markersize=4, label='Validation Loss', markeredgecolor='black', markeredgewidth=0.5)

    best_epoch = val_losses.index(min(val_losses)) + 1
    ax.axvline(x=best_epoch, color=blues['gray'], linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'{MODEL_NAME} - Training and Validation Loss')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_facecolor('#f8f9fa')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/loss_curves.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    # GRÁFICO 2: Accuracy
    print("Gráfico 2: Accuracy...")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, val_accuracies, 'o-', color=blues['medium'], linewidth=1.5,
            markersize=4, label='Validation Accuracy', markeredgecolor='black', markeredgewidth=0.5)
    best_acc = max(val_accuracies)
    ax.axhline(y=best_acc, color=blues['gray'], linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{MODEL_NAME} - Validation Accuracy')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim([0, 1.05])
    ax.set_facecolor('#f8f9fa')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/validation_accuracy.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    # GRÁFICO 3: Curva ROC
    print("Gráfico 3: Curva ROC...")
    fpr, tpr, _ = roc_curve(test_labels, test_probs)
    roc_auc = test_auc
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(fpr,tpr,color=blues["medium"],linewidth=2,label=f"{MODEL_NAME} (AUC = {roc_auc:.3f})")
    ax.plot([0, 1], [0, 1], color=blues['gray'], linestyle='--', linewidth=1, label='Random Classifier')
    ax.fill_between(fpr, tpr, alpha=0.15, color=blues['medium'])
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{MODEL_NAME} - ROC Curve')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_facecolor('#f8f9fa')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/roc_curve.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    # GRÁFICO 4: Matriz de Confusión
    print("Gráfico 4: Matriz de confusión...")
    cm = cm_test
    cm_percent = cm / cm.sum() * 100
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                ax=ax, cbar_kws={'label': 'Count', 'shrink': 0.8})
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.7, f'({cm_percent[i,j]:.1f}%)',
                    ha='center', va='center', fontsize=9, color='black')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(f'{MODEL_NAME} - Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/confusion_matrix.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    # GRÁFICO 5: Barras de Métricas
    print("Gráfico 5: Barras de métricas...")
    metrics = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1-Score', 'AUC-ROC']
    values = [test_acc, test_recall, test_specificity, test_precision, test_f1, test_auc]
    # Gradiente de azules
    blue_gradient = [blues['dark'], blues['medium'], blues['medium'], blues['light'], blues['lighter'], blues['lightest']]
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(metrics, values, color=blue_gradient, edgecolor='black', linewidth=0.8)
    ax.axhline(y=0.85, color=blues['gray'], linestyle='--', linewidth=1, alpha=0.7, label='Clinical Threshold (85%)')
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.set_title(f'{MODEL_NAME} - Performance Metrics')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.set_facecolor('#f8f9fa')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/metrics_bars.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    # GRÁFICO 6: Distribución de Probabilidades
    print("Gráfico 6: Distribución de probabilidades...")
    pos_probs = [test_probs[i] for i in range(len(test_labels)) if test_labels[i] == 1]
    neg_probs = [test_probs[i] for i in range(len(test_labels)) if test_labels[i] == 0]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(neg_probs, bins=30, alpha=0.6, color=blues['light'], label=f'Healthy (n={len(neg_probs)})',
            density=True, edgecolor='black', histtype='stepfilled')
    ax.hist(pos_probs, bins=30, alpha=0.8, color=blues['dark'], label=f'Cancer (n={len(pos_probs)})',
            density=True, edgecolor='black', histtype='stepfilled')
    ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold (0.5)')
    ax.set_xlabel('Predicted Probability of Cancer')
    ax.set_ylabel('Density')
    ax.set_title(f'{MODEL_NAME} - Probability Distribution')
    ax.legend(loc='upper center')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_facecolor('#f8f9fa')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/probability_distribution.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    # GRÁFICO 7: Curva Precisión-Recall
    print("Gráfico 7: Curva Precisión-Recall...")
    precision_curve, recall_curve, _ = precision_recall_curve(test_labels, test_probs)
    avg_precision = average_precision_score(test_labels, test_probs)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(recall_curve, precision_curve, color=blues['medium'], linewidth=2, label=f"{MODEL_NAME} (AP = {avg_precision:.3f})")
    ax.fill_between(recall_curve, precision_curve, alpha=0.15, color=blues['medium'])
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall (Sensitivity)')
    ax.set_ylabel('Precision (PPV)')
    ax.set_title(f'{MODEL_NAME} - Precision-Recall Curve')
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_facecolor('#f8f9fa')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/plots/{MODEL_NAME}/precision_recall_curve.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

# MOBILENETV2

In [ ]:
MODEL_NAME = "MobileNetV2"

def create_mobilenetv2():
    model = models.mobilenet_v2(
        weights=MobileNet_V2_Weights.IMAGENET1K_V1
    )

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(in_features, 1)
    )

    return model


class MobileNetV2Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = create_mobilenetv2()

    def forward(self, x):
        return self.backbone(x).squeeze(1)

## Crear el modelo

In [ ]:
model = MobileNetV2Classifier().to(DEVICE)

print(f"Modelo: {MODEL_NAME}")
print(f"Total parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"Entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Probar Forward
x = torch.randn(2,3,224,224).to(DEVICE)
y = model(x)

print("Input:",x.shape)
print("Output:",y.shape)

## Configuración de entrenamiento 1

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

EARLY_STOPPING = 7

print("Configuración")

print("Learning Rate:",LEARNING_RATE)
print("Epochs:",NUM_EPOCHS)
print("Batch Size:",BATCH_SIZE)

## Función Train

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*images.size(0)
        preds = (torch.sigmoid(outputs)>0.5).float()
        correct += (preds==labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss/len(loader.dataset)
    epoch_acc = correct/total

    return epoch_loss, epoch_acc

## Validation

In [ ]:
def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()*images.size(0)
            probs = torch.sigmoid(outputs)
            preds = (probs>0.5).float()
            correct += (preds==labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss/len(loader.dataset)
    epoch_acc = correct/total

    return epoch_loss, epoch_acc, all_preds, all_labels, all_probs

## Entrenamiento 2

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss, val_acc, preds, labels, probs = validate_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    precision = precision_score(labels,preds)
    recall = recall_score(labels,preds)
    f1 = f1_score(labels,preds)
    cm = confusion_matrix(labels,preds)
    tn,fp,fn,tp = cm.ravel()
    specificity = tn/(tn+fp)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1: {f1:.4f}")

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        torch.save(
            model.state_dict(),
            f"{SAVE_DIR}/models/best_{MODEL_NAME}.pth"
        )

    else:
        patience_counter += 1

        if patience_counter >= EARLY_STOPPING:
            print("Early Stopping")
            break

## Evaluación 3

In [ ]:
model.load_state_dict(
    torch.load(f"{SAVE_DIR}/models/best_{MODEL_NAME}.pth")
)

test_loss, test_acc, test_preds, test_labels, test_probs = validate_epoch(
    model,
    test_loader,
    criterion,
    DEVICE
)

print(f"\nResultados {MODEL_NAME}")

test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall = recall_score(test_labels, test_preds, zero_division=0)
test_f1 = f1_score(test_labels, test_preds, zero_division=0)
test_auc = roc_auc_score(test_labels, test_probs)

cm_test = confusion_matrix(test_labels, test_preds)

tn, fp, fn, tp = cm_test.ravel()

test_specificity = tn / (tn + fp)

print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"Specificity: {test_specificity:.4f}")
print(f"F1: {test_f1:.4f}")
print(f"AUC: {test_auc:.4f}")

# DENSENET121

In [ ]:
# Limpiar memoria antes de entrenar nuevo modelo
torch.cuda.empty_cache()
import gc
gc.collect()

MODEL_NAME = "DenseNet121"
class DenseNet121Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.densenet121(
            weights=DenseNet121_Weights.IMAGENET1K_V1
        )
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, 1)
        )

    def forward(self, x):
        return self.backbone(x).squeeze(1)

## Crear el modelo

In [ ]:
model = DenseNet121Classifier().to(DEVICE)

print(f"Modelo: {MODEL_NAME}")
print(f"Total parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"Entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Probar forward pass
test_input = torch.randn(2, 3, 224, 224).to(DEVICE)
test_output = model(test_input)
print(f"\nTest forward: input shape {test_input.shape} -> output shape {test_output.shape}")

despues de este ejecuto las de arriba con número

# EFFICIENTNET-B0

In [ ]:
# Limpiar memoria antes de entrenar nuevo modelo
torch.cuda.empty_cache()

import gc
gc.collect()

MODEL_NAME = "EfficientNetB0"

class EfficientNetB0Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.efficientnet_b0(
            weights=EfficientNet_B0_Weights.IMAGENET1K_V1
        )

        in_features = self.backbone.classifier[1].in_features

        self.backbone.classifier[1] = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, 1)
        )

    def forward(self, x):
        return self.backbone(x).squeeze(1)

## Crear el modelo

In [ ]:
model = EfficientNetB0Classifier().to(DEVICE)

print(f"Modelo: {MODEL_NAME}")
print(f"Total parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"Entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

test_input = torch.randn(2,3,224,224).to(DEVICE)
test_output = model(test_input)

print(f"\nTest forward:")
print(f"Input : {test_input.shape}")
print(f"Output: {test_output.shape}")

despues de este ejecuto las de arriba con número

# ResNet18 + LSTM

In [ ]:
torch.cuda.empty_cache()
gc.collect()

MODEL_NAME = "ResNet18_LSTM"

class ResNet18LSTM(nn.Module):
    def __init__(self, num_classes=1, lstm_hidden=256, num_layers=2, dropout=0.2):
        super(ResNet18LSTM, self).__init__()

        # ResNet18 pre-entrenado (sin el clasificador final)
        self.resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()  # Eliminar clasificador, solo características

        # LSTM para procesar secuencias
        self.lstm = nn.LSTM(
            input_size=in_features,  # 512 features de ResNet
            hidden_size=lstm_hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )

        # Capa final de clasificación
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.shape
        x = x.view(batch_size * seq_len, C, H, W)
        features = self.resnet(x)
        features = features.view(batch_size, seq_len, -1)
        lstm_out, (hidden, cell) = self.lstm(features)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output.squeeze(1)

### Después de cada entrenamiento 4

In [ ]:
results = {
    "model": MODEL_NAME,
    "accuracy": test_acc,
    "precision": test_precision,
    "recall": test_recall,
    "specificity": test_specificity,
    "f1_score": test_f1,
    "auc_roc": test_auc,
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp)
}

df_results = pd.DataFrame([results])

df_results.to_csv(
    f"{SAVE_DIR}/results/{MODEL_NAME}.csv",
    index=False
)

with open(
    f"{SAVE_DIR}/results/{MODEL_NAME}_results.json",
    "w"
) as f:
    json.dump(results, f, indent=4)

print(f"\nResultados de {MODEL_NAME} guardados correctamente.")
print(f"CSV  : {MODEL_NAME}.csv")
print(f"JSON : {MODEL_NAME}_results.json")

In [ ]:
generar_graficos(
    MODEL_NAME=MODEL_NAME,
    train_losses=train_losses,
    val_losses=val_losses,
    val_accuracies=val_accuracies,
    test_labels=test_labels,
    test_preds=test_preds,
    test_probs=test_probs,
    test_acc=test_acc,
    test_recall=test_recall,
    test_specificity=test_specificity,
    test_precision=test_precision,
    test_f1=test_f1,
    test_auc=test_auc,
    cm_test=cm_test,
    SAVE_DIR=SAVE_DIR
)